Fine-tuning libraries installation.

In [5]:
!!pip install -U --upgrade unsloth unsloth_zoo datasets transformers accelerate bitsandbytes peft trl huggingface-hub

['Looking in indexes: http://cor-notebook-dev-wheel-cache.projects:8081/simple, https://download.pytorch.org/whl/cu130',
 'Looking in links: /var/cache/pip/wheels-local, /var/cache/pip/wheels',
 'Requirement already satisfied: unsloth in /home/coder/python-packages/lib/python3.12/site-packages (2026.3.17)',
 'Requirement already satisfied: unsloth_zoo in /home/coder/python-packages/lib/python3.12/site-packages (2026.3.6)',
 'Requirement already satisfied: datasets in /home/coder/python-packages/lib/python3.12/site-packages (4.3.0)',
 'Collecting datasets',
 '  Downloading http://cor-notebook-dev-wheel-cache.projects:8081/files/datasets/datasets-4.8.4-py3-none-any.whl (526 kB)',
 '\x1b[?25l     \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b0.0/527.0 kB\x1b \x1b?\x1b eta \x1b-:--:--\x1b',
 '\x1b[2K     \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b527.0/527.0 kB\x1b \x1b430.5 MB/s\x1b eta \x1b0:00:00\x1b',
 '\x1b[?25hRequirement already satisfied: transformers in /home/coder/p

Imports.

In [8]:
from datasets import load_dataset, concatenate_datasets, DatasetDict
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch

In [4]:
!pip install https://github.com/vllm-project/vllm/releases/download/v0.13.0/vllm-0.13.0+cu130-cp38-abi3-manylinux_2_35_x86_64.whl

Looking in indexes: http://cor-notebook-dev-wheel-cache.projects:8081/simple, https://download.pytorch.org/whl/cu130
Looking in links: /var/cache/pip/wheels-local, /var/cache/pip/wheels
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 37.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 235.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 114.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 322.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 128.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 MB 125.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.4/419.4 MB 124.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 126.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 125.9 MB/s eta 0:00:0000:0100:01
     ━━━━

Dataset download.

In [9]:
hf_dataset_id = "pymlex/future-engineers-dataset"
max_seq_length = 4096
raw = load_dataset(hf_dataset_id)

Choosing only short examples.

In [10]:
ds = concatenate_datasets([raw[k] for k in raw.keys()])
ds = ds.filter(lambda x: isinstance(x["text"], str) and len(x["text"]) < 8000)

split = ds.train_test_split(test_size=0.1, seed=3407)
split2 = split["test"].train_test_split(test_size=0.5, seed=3407)

Filter: 100%|██████████| 2193/2193 [00:00<00:00, 54417.02 examples/s]


Hold-out with 90/5/5 ratio.

In [11]:
train_base = split["train"]
val_base = split2["train"]
test_base = split2["test"]

unsloth/Qwen3-4B model initialisation

In [20]:
model_name = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0. vLLM: 0.13.0+cu130.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 398/398 [00:01<00:00, 320.40it/s]


unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)

==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0. vLLM: 0.13.0+cu130.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 427.90it/s]


unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.17 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Creating chat template.

In [13]:
system_prompt = """
Ты пишешь полные научно-технические описания школьных инженерных и
ИТ-проектов для конференции. Пиши по-русски, связно, подробно, конкретно,
уверенно и без воды. Не сокращай текст до резюме и общего видения.
Требуется аннотация, введение с актуальностью, цель, задачи,
оборудование (конкретное, с названиями) и программы (с названиями),
ход работы, выводы, перспективы.
"""

def format_example(example):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Тема: {example['topic']}. Подготовь полное описание проекта для конференции «Инженеры будущего»."},
        {"role": "assistant", "content": example["text"]}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )
    n_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return {"text": text, "n_tokens": n_tokens}

Removing unnecessary properties from samples.

In [14]:
train_ds = train_base.map(format_example, remove_columns=train_base.column_names)
val_ds = val_base.map(format_example, remove_columns=val_base.column_names)
test_ds = test_base.map(format_example, remove_columns=test_base.column_names)

train_ds = train_ds.filter(lambda x: x["n_tokens"] <= max_seq_length)
val_ds = val_ds.filter(lambda x: x["n_tokens"] <= max_seq_length)
test_ds = test_ds.filter(lambda x: x["n_tokens"] <= max_seq_length)

train_ds = train_ds.remove_columns(["n_tokens"])
val_ds = val_ds.remove_columns(["n_tokens"])
test_ds = test_ds.remove_columns(["n_tokens"])

bf16 = torch.cuda.is_bf16_supported()
fp16 = not bf16

Filter: 100%|██████████| 107/107 [00:00<00:00, 27046.98 examples/s]


Setting trainer and its config.

In [21]:
train_args = SFTConfig(
    output_dir="/home/coder/project/jupyter/qwen3_future_engineers_lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=1.5e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    bf16=bf16,
    fp16=fp16,
    report_to="none",
    seed=3407
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,
    args=train_args
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Training. The model was trained on RTX 4090 GPU (24 GB VRAM) with 13 CPUs and 15 GB of RAM. The first two epochs were run with a learning rate of 1.5e-4 (40 min), while for the third epoch, it was reduced to 0.5e-4 (20 min). The entire process took one hour. By step 10, the loss was approximately 2.0. By step 200, it had dropped to about 1.0. During the third epoch, it decreased further to 0.95.

In [ ]:
trainer.train()

In [24]:
trainer.save_model("/home/coder/project/jupyter/qwen3_future_engineers_lora")
tokenizer.save_pretrained("/home/coder/project/jupyter/qwen3_future_engineers_lora")

('/home/coder/project/jupyter/qwen3_future_engineers_lora/tokenizer_config.json',
 '/home/coder/project/jupyter/qwen3_future_engineers_lora/chat_template.jinja',
 '/home/coder/project/jupyter/qwen3_future_engineers_lora/tokenizer.json')

Saving the model to Hugging Face.

In [ ]:
!hf auth login --token YOUR_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `future-engineers` has been saved to /home/coder/.cache/huggingface/stored_tokens
Your token has been saved to /home/coder/.cache/huggingface/token
Login successful.
The current active token is: `future-engineers`


In [29]:
model.push_to_hub("pymlex/qwen3-future-engineers-lora")
tokenizer.push_to_hub("pymlex/qwen3-future-engineers-lora")

Processing Files (1 / 1): 100%|██████████|  132MB /  132MB, 16.9MB/s  
New Data Upload: 100%|██████████|  132MB /  132MB, 16.9MB/s  


Saved model to https://huggingface.co/pymlex/qwen3-future-engineers-lora


Processing Files (1 / 1): 100%|██████████| 11.4MB / 11.4MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Inference.

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model = "Qwen/Qwen3-4B-Instruct-2507"
adapter_path = "pymlex/qwen3-future-engineers-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

/home/coder/python-packages/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 305.96it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [2]:
system_prompt = """
Ты пишешь полные научно-технические описания школьных инженерных и
ИТ-проектов для конференции. Пиши по-русски, связно, подробно, конкретно,
уверенно и без воды. Не сокращай текст до резюме и общего видения.
Требуется аннотация, введение с актуальностью, цель, задачи,
оборудование (конкретное, с названиями) и программы (с названиями),
ход работы, выводы, перспективы.
"""

def generate(topic):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Тема: {topic}. Подготовь полное описание проекта для конференции «Инженеры будущего»."}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_len = inputs.input_ids.shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=1200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    decoded = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    print(decoded.strip())

In [5]:
generate("Система мониторинга качества воздуха в школьных кабинетах")

</tool_call>

<tool_call>

# Актуальность

В настоящее время все больше людей страдают от проблем, вызванных низким качеством воздуха. Здоровье людей зависит от многих факторов, но одним из важнейших является качество воздуха. В нашей школе есть кабинеты с очень плохой вентиляцией. Поэтому мы решили создать систему мониторинга качества воздуха, которая будет показывать уровень углекислого газа и влажности воздуха в каждом кабинете нашей школы.

# Цель

Разработать и создать систему мониторинга качества воздуха в кабинетах нашей школы.

# Задачи

- 1. Изучить теоретический материал по теме.
- 2. Изучить существующие аналоги.
- 3. Разработать и создать систему мониторинга качества воздуха.
- 4. Провести анализ работы созданной системы.

# Оснащение и оборудование, использованное при создании работы

- 1. Микроконтроллер Arduino UNO
- 2. Датчик влажности воздуха DHT11
- 3. Датчик концентрации углекислого газа MQ135
- 4. Адаптер питания USB
- 5. Компьютер с установленным ПО: Arduino IDE, F

In [4]:
generate("Приложение для автоматизации учета домашних заданий с использованием ИИ")

</tool_call>

<tool_call>

# Актуальность

В современном мире всё большую роль играют информационные технологии, которые позволяют оптимизировать многие процессы. Одной из таких областей, где они могут быть применены, является образование. В школе ученики часто сталкиваются с трудностями в запоминании и отслеживании домашних заданий. Это может привести к низкой успеваемости и снижению мотивации к обучению. Для решения этой проблемы можно использовать приложение, которое автоматически уведомляет учеников о новых заданиях и помогает им отслеживать выполнение домашних заданий.

# Цель

Разработка приложения, которое поможет ученикам автоматически отслеживать и уведомлять о новых домашних заданиях.

# Задачи

- 1. Изучить существующие решения для автоматизации учёта домашних заданий.
- 2. Разработать концепцию приложения.
- 3. Создать базу данных для хранения информации о домашних заданиях.
- 4. Разработать интерфейс пользователя для удобного взаимодействия с приложением.
- 5. Реализовать 

In [9]:
generate("Разработка умной теплицы с автоматическим контролем влажности и температуры")

</tool_call>

<tool_call>

# **Актуальность**

В настоящее время ведутся активные разработки по улучшению качества жизни людей и повышению их комфорта. Одним из направлений, в котором проявляется стремление улучшить качество жизни, является создание умных пространств. Умные пространства включают в себя элементы автоматизации, которые позволяют управлять различными устройствами и системами. Одним из примеров использования умных пространств является создание умной теплицы. Умная теплица может обеспечивать комфортные условия для роста растений, автоматически регулируя такие параметры, как влажность и температура. Она может быть оснащена датчиками, которые постоянно отслеживают изменения влажности и температуры, и автоматически регулировать уровень влажности и температуры в соответствии с необходимыми параметрами. Это может включать в себя включение или отключение влажности и температуры. Умная теплица может также иметь встроенные камеры, которые позволяют пользователям наблюдать за ростом

In [8]:
generate("Физически информированная нейронная сеть для моделирования движения ледокола")

</tool_call>

<tool_call>

# **Актуальность**

В настоящее время ледоколы используются для обеспечения проходимости воды в зимний период, что необходимо для поддержания транспортных и энергетических связей. Важным аспектом использования ледоколов является их эффективное движение по льду, что позволяет достигать максимальной скорости передвижения и минимизировать затраты на энергию. Одним из ключевых факторов, влияющих на движение ледокола, является форма его корпуса. Угол наклона кормы играет значительную роль в определении траектории движения ледокола. Угол наклона кормы влияет на движение ледокола, поскольку он определяет траекторию движения ледокола. В результате, при небольшом наклоне кормы ледокол движется более плавно, а при большем наклоне ледокол может потерять контроль над своим движением и потерять стабильность.

# **Цель**

Создание физически информированной нейронной сети, которая может моделировать движение ледокола и определять оптимальный угол наклона кормы для достижени